In [2]:
from deepeval.metrics import BaseMetric
from typing import Dict, Any, List, Optional, Union, Tuple

class CustomBaseMetric(BaseMetric):
    """Extended base class for custom metrics compatible with DeepEval"""
    
    def __init__(self, 
                 name: str, 
                 description: str, 
                 threshold: float = None,
                 csv_requires: List[str] = None,
                 runtime_requires: List[str] = None):
        """
        Initialize a custom metric that extends DeepEval's BaseMetric
        
        Args:
            name: Name of the metric
            description: Description of what the metric measures
            threshold: Optional threshold for passing/failing
            csv_requires: Required fields from CSV data
            runtime_requires: Required parameters at runtime
        """
        # Initialize BaseMetric attributes properly without calling super().__init__
        # as that would cause issues with the non-standard parameters
        self.threshold = threshold
        
        # Set our custom attributes
        self.name = name
        self.description = description
        self.csv_requires = csv_requires if csv_requires else []
        self.runtime_requires = runtime_requires if runtime_requires else []
        self._score = None
        
    def _is_successful(self) -> bool:
        """
        Returns True if the metric passes the threshold
        Required by DeepEval's BaseMetric
        """
        if self.threshold is None or self._score is None:
            return True
        return self._score >= self.threshold
    
    def _metric_name(self) -> str:
        """
        Returns the name of the metric
        Required by DeepEval's BaseMetric
        """
        return self.name
    
    def _metric_value(self) -> Union[float, int, bool, str, Dict, List, Tuple, None]:
        """
        Returns the metric value
        Required by DeepEval's BaseMetric
        """
        return self._score
    
    @property
    def score(self) -> Union[float, int, bool, str, Dict, List, Tuple, None]:
        """
        Returns the score for easy access
        """
        return self._score
    
    def is_successful(self) -> bool:
        """
        Returns whether the metric passed the threshold
        """
        return self._is_successful()
    
    def calculate(self, **kwargs) -> Dict[str, Any]:
        """
        Calculate metric from required inputs
        Must be implemented by child classes
        """
        raise NotImplementedError("Subclasses must implement calculate()")
    
    def measure(self, **kwargs) -> None:
        """
        Measure the metric and set the score
        Required by DeepEval's BaseMetric
        """
        result = self.calculate(**kwargs)
        if isinstance(result, dict) and 'score' in result:
            self._score = result['score']
        elif isinstance(result, (int, float)):
            self._score = result
        else:
            # Depending on your metric, you may need to adapt this logic
            try:
                main_key = next(iter(result)) if result else None
                self._score = result[main_key] if main_key else 0.0
            except:
                self._score = 0.0

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from typing import Dict, Any
import asyncio

class SQLClauseCountGEval(CustomBaseMetric):
    """LLM-based clause count matching using DeepEval's GEval"""
    
    def __init__(self, threshold: float = 1.0):
        CustomBaseMetric.__init__(
            self,
            name="clause_count_match",
            description="Checks if generated SQL has same number of clauses as reference SQL",
            threshold=threshold,
            csv_requires=["gold_sql"],
            runtime_requires=["generated_sql", "gold_sql"]
        )
        
        # Initialize GEval with custom criteria
        self.geval = GEval(
            name="ClauseCountMatch",
            criteria="Verify if the generated SQL uses the same number of clauses (SELECT, FROM, WHERE, etc.) as the reference SQL.",
            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
            evaluation_steps=[
                "Identify all clauses in the reference SQL (input)",
                "Count clauses in the generated SQL (actual_output)",
                "Compare the counts of each clause type",
                
            ],
            model="gpt-4o-mini"
        )

    async def measure_async(self, generated_sql: str, gold_sql: str) -> None:
        """Async version of measure for proper await handling"""
        test_case = LLMTestCase(
            input=gold_sql,
            actual_output=generated_sql
        )
        
        try:
            # Run async evaluation
            await self.geval.a_measure(test_case)
            self._process_score()
        except Exception as e:
            print(f"Evaluation error: {str(e)}")
            self._score = 0.0

    def measure(self, generated_sql: str, gold_sql: str) -> None:
        """Sync wrapper for async measure"""
        asyncio.run(self.measure_async(generated_sql, gold_sql))

    def _process_score(self):
        """Handle score conversion consistently"""
        try:
            raw_score = self.geval.score
            if isinstance(raw_score, (int, float)):
                self._score = float(raw_score)
            elif isinstance(raw_score, str):
                clean_score = raw_score.strip().lower()
                self._score = 1.0 if clean_score in ["1", "yes"] else 0.0
            else:
                self._score = 0.0
        except Exception as e:
            print(f"Score processing error: {str(e)}")
            self._score = 0.0

    def calculate(self, generated_sql: str, gold_sql: str) -> Dict[str, Any]:
        """Calculate method now uses sync interface"""
        self.measure(generated_sql, gold_sql)
        return {
            "score": self._score,
            "generated_sql": generated_sql,
            "gold_sql": gold_sql,
            "evaluation_rationale": self.geval.reason
        }

In [7]:
# Ensure clause_metric is defined first
clause_metric = SQLClauseCountGEval(threshold=1.0)

# Wrap the test logic inside an async function so you can use `await`
async def test_with_deepeval():
    queries = [
        { 
            "input": "Count employees over 56 years old",
            "generated_sql": "SELECT count(*) FROM head WHERE age  >  56",
            "gold_sql": "SELECT count(*) FROM head WHERE age  >  56",
            "db_id": "department_management"
        },
        {
            "input": "Find most common budget type in documents with expenses",
            "generated_sql": "SELECT Budget_Type_Code, COUNT(*) AS Count FROM Documents_with_Expenses GROUP BY Budget_Type_Code ORDER BY Count DESC LIMIT 1;",
            "gold_sql": "SELECT budget_type_code FROM Documents_with_expenses GROUP BY budget_type_code ORDER BY count(*) DESC LIMIT 1",
            "db_id": "cre_Docs_and_Epenses"
        }
    ]
    
    print("=" * 60)
    print("SQL METRICS EVALUATION")
    print("=" * 60)

    for i, query in enumerate(queries):
        print(f"\nQuery Pair #{i+1}:")
        print(f"Input: {query['input']}")
        print(f"Generated: {query['generated_sql']}")
        print(f"Gold: {query['gold_sql']}")
        print("-" * 50)

        try:
            await clause_metric.measure_async(
                generated_sql=query['generated_sql'],
                gold_sql=query['gold_sql']
            )
            llm_result = {
                "score": clause_metric.score,
                "passed": clause_metric.is_successful(),
                "rationale": clause_metric.geval.reason
            }
            print(f"LLM Clause Match: {llm_result}")
        except Exception as e:
            print(f"LLM Clause Match Error: {str(e)}")

    print("-" * 50)

In [8]:
await test_with_deepeval()

c:\Users\AnmolSehgal\miniconda3\envs\llm_benchmarking\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

SQL METRICS EVALUATION

Query Pair #1:
Input: Count employees over 56 years old
Generated: SELECT count(*) FROM head WHERE age  >  56
Gold: SELECT count(*) FROM head WHERE age  >  56
--------------------------------------------------


LLM Clause Match: {'score': 1.0, 'passed': True, 'rationale': 'The actual output matches the input exactly, with the same clause count and type.'}

Query Pair #2:
Input: Find most common budget type in documents with expenses
Generated: SELECT Budget_Type_Code, COUNT(*) AS Count FROM Documents_with_Expenses GROUP BY Budget_Type_Code ORDER BY Count DESC LIMIT 1;
Gold: SELECT budget_type_code FROM Documents_with_expenses GROUP BY budget_type_code ORDER BY count(*) DESC LIMIT 1
--------------------------------------------------


LLM Clause Match: {'score': 0.7555312974501841, 'passed': False, 'rationale': 'The actual output includes an additional COUNT(*) clause and an alias, which enhances the query but slightly deviates from the original intent of only selecting the budget_type_code.'}
--------------------------------------------------
